## ex03_aggs.ipynb

In [34]:
import pandas as pd
import sqlite3

## Create a connection to the database using the sqlite3 library.

In [35]:
db_path = "../data/checking-logs.sqlite"
connection = sqlite3.connect(db_path)

## Get the schema of the test table.

In [36]:
query = "PRAGMA table_info(test)"
schema = pd.read_sql(query, connection)
display(schema)

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,uid,TEXT,0,None,0
2,2,labname,TEXT,0,None,0
3,3,first_commit_ts,TIMESTAMP,0,None,0
4,4,first_view_ts,TIMESTAMP,0,None,0


## Get only the first ten rows of the test table to see what it looks like.

In [37]:
query = "SELECT * FROM test LIMIT 10"
first_ten_rows = pd.io.sql.read_sql(query, connection)
display(first_ten_rows)

,index,uid,labname,first_commit_ts,first_view_ts
0,0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


## Find the minimum value of the delta between the first commit and the deadline of the corresponding lab for all users using only one query.

In [38]:
query = """
        SELECT uid, (unixepoch(first_commit_ts) - deadlines)/3600 AS delta
        FROM test
            JOIN deadlines ON  test.labname = deadlines.labs
        WHERE labs <> 'project1'
        ORDER BY delta ASC
        LIMIT 1
        """
df_min = pd.io.sql.read_sql(query, connection)
display(df_min)

,uid,delta
0,user_30,-202


## Do the same thing for the maximum, but use only one query. The dataframe name is df_max.

In [39]:
query = """
        SELECT uid, (unixepoch(first_commit_ts) - deadlines)/3600 AS delta
        FROM test
            JOIN deadlines ON  test.labname = deadlines.labs
        WHERE labs <> 'project1'
        ORDER BY delta DESC
        LIMIT 1
        """
df_max = pd.io.sql.read_sql(query, connection)
display(df_max)

,uid,delta
0,user_25,-2


## Do the same thing, but for the average. Use only one query. This time, your dataframe should not include the uid column. The dataframe name is df_avg.

In [40]:
query = """
        SELECT AVG(diff)
        FROM
            (SELECT uid, CAST((JULIANDAY(DISTINCT first_commit_ts) - JULIANDAY(DATETIME(deadlines, 'unixepoch'))) * 24 AS INTEGER) AS diff
            FROM test
            LEFT JOIN deadlines
            ON test.labname=deadlines.labs
            WHERE test.labname <> "project1"
            GROUP BY uid, test.labname)
        """
df_avg = pd.io.sql.read_sql(query, connection)
display(df_avg)

,AVG(diff)
0,-89.125


## We want to test the hypothesis that users who visited the newsfeed just a few times have a lower delta between the first commit and the deadline. To do this, calculate the correlation coefficient between the number of pageviews and the difference.

In [41]:
query = """
        SELECT test.uid, AVG(unixepoch(first_commit_ts)-deadlines) AS avg_diff,
            views AS pageviews
        FROM test
            JOIN deadlines ON test.labname = deadlines.labs
            JOIN (
                SELECT pageviews.uid, COUNT(datetime) AS views
                FROM pageviews
                GROUP BY uid
            ) AS pageviews_per_user ON test.uid = pageviews_per_user.uid
        WHERE labs <> 'project1'
        GROUP BY test.uid
        """

views_diff = pd.read_sql(query, connection)
display(views_diff[['avg_diff','pageviews']].corr().round(6))

,avg_diff,pageviews
avg_diff,1.000000,-0.279143
pageviews,-0.279143,1.000000


## Close the connection.

In [42]:
connection.close()